#  eCP-FS on ccnews-nomic-768



**How to run it**
1. Set `DRY_RUN = True` and do *Restart Kernel → Run All*. This smoke test builds a tiny index from 20,000 vectors, runs 5 queries at 2 settings, and goes through every stage in a minute or two.
2. Check that the output looks sensible (no errors, files written, numbers plausible).
3. Set `DRY_RUN = False` and *Restart → Run All* again for the real run.

Every run writes into its own folder, `results/<run_id>/` (dry runs: `results/dry_<run_id>/`), with `config.json`, the raw per-query results, a summary table and the plots.

In [ ]:
# ---- Settings: the only cell you normally edit ----
DRY_RUN = False

DATASET = "ccnews-nomic-768-normalized"
EXPECTED_TRAIN_SHAPE = (495_328, 768)       # sanity checks for this dataset
EXPECTED_N_TEST = 1000

LEVELS = 3
TARGET_CLUSTER_ITEMS = 43
STRATEGY = "random"
K = 10

SWEEP = [1, 2, 4, 8, 16, 32]       # search_exp values to test
N_QUERIES = 200                            # None = all test queries
REPEATS = 1                                 # use 3 on the cluster
SEARCH_MEMORY_LIMIT = 3 * 1024**3           # cap on eCP-FS's cache while searching
BUILD_MEMORY_LIMIT = 3 * 1024**3            # memory target for the build
EVICT_OS_CACHE = True                       # Linux only: true cold start before each fresh pass
RUN_CACHE_TEST = True                     # optional diagnostic, stage 2b

if DRY_RUN:                                 # smoke test: the whole pipeline in about a minute
    SWEEP, N_QUERIES, REPEATS = [1, 2], 5, 1
    DRY_TRAIN_ROWS = 20_000

In [ ]:
import gc, json, os, platform, shutil, subprocess, threading, time
from datetime import datetime
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from ecpfs import Builder, Index, Metric

# Repo root = the first folder upwards that contains pyproject.toml,
# so the notebook works wherever it is saved .
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FULL_DATA = REPO / "data" / f"{DATASET}.h5"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = REPO / "results" / (("dry_" if DRY_RUN else "") + RUN_ID)
OUT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID, "dry_run": DRY_RUN, "dataset": DATASET,
    "levels": LEVELS, "target_cluster_items": TARGET_CLUSTER_ITEMS, "strategy": STRATEGY,
    "metric": "L2", "k": K, "sweep": SWEEP, "n_queries": N_QUERIES, "repeats": REPEATS,
    "search_memory_limit_bytes": SEARCH_MEMORY_LIMIT,
    "build_memory_limit_bytes": BUILD_MEMORY_LIMIT,
}

def save_config():
    # Writes CONFIG (and ENV, once it exists) to the run folder. Called again whenever CONFIG grows.
    payload = {"config": CONFIG, "env": globals().get("ENV", {})}
    (OUT / "config.json").write_text(json.dumps(payload, indent=2, default=str))

print("Repo:      ", REPO)
print("Data found:", FULL_DATA.exists())
print("Results:   ", OUT)

In [ ]:
#  Measurement helpers ----

def git_commit(repo):
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"

def ram_mb():
    # Memory reserved by this process, in MB: 'private bytes' on Windows (the working set
    # under-reports there), RSS on Linux
    mi = psutil.Process().memory_info()
    return round(getattr(mi, "private", mi.rss) / 1e6)

class PeakMemory:
    # Samples ram_mb() in a background thread; keeps the start value and the peak.
    # Works because ecpfs releases Python's lock while it builds and searches.
    def __init__(self, interval=0.2):
        self.interval = interval

    def __enter__(self):
        self.start_mb = self.peak_mb = ram_mb()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def _run(self):
        while not self._stop.is_set():
            self.peak_mb = max(self.peak_mb, ram_mb())
            self._stop.wait(self.interval)

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join()
        self.peak_mb = max(self.peak_mb, ram_mb())

    @property
    def added_mb(self):
        return self.peak_mb - self.start_mb

def folder_size(path):
    files = [p for p in Path(path).rglob("*") if p.is_file()]
    return sum(p.stat().st_size for p in files), len(files)

def page_faults():
    # Windows: soft + hard page faults so far; 0 on other systems
    return getattr(psutil.Process().memory_info(), "num_page_faults", 0)

def evict_os_cache(path):
    # Linux only: ask the OS to drop the index files from its page cache (no root needed).
    # Returns False where unsupported (Windows), so 'fresh' there means 'fresh process', not cold disk.
    if not hasattr(os, "posix_fadvise"):
        return False
    for p in Path(path).rglob("*"):
        if p.is_file():
            fd = os.open(p, os.O_RDONLY)
            try:
                os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
            finally:
                os.close(fd)
    return True

## Stage 1 · Get data

 Each news article was turned into 768 numbers by an embedding model, so similar articles are close together. We check the file before using it (*garbage in, garbage out*): if the vectors weren't unit length, dot product and L2 would rank neighbours differently, and every later step would be wrong. Then we build the eCP-FS index: 11,520 random *leaders*, every vector joins its nearest leader (a leaf of ~43 vectors ≈ one 128 KB disk read), and the leaders are grouped into a 3-level tree.

In [ ]:
# Record the machine and code version: a latency number is meaningless without them
ENV = {
    "git_commit": git_commit(REPO),
    "python": platform.python_version(),
    "os": platform.platform(),
    "hostname": platform.node(),
    "cpu": platform.processor() or platform.machine(),
    "physical_cores": psutil.cpu_count(logical=False),
    "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
}
save_config()
ENV

In [ ]:
# Inspect the full dataset and check it is the one we expect
with h5py.File(FULL_DATA, "r") as f:
    print("File attributes:", dict(f.attrs))
    for name, ds in f.items():
        print(f"{name:14s} shape={ds.shape}  dtype={ds.dtype}")
    n_train, dim = f["train"].shape
    n_test = f["test"].shape[0]

assert (n_train, dim) == EXPECTED_TRAIN_SHAPE, "unexpected train shape"
assert n_test == EXPECTED_N_TEST, "unexpected number of queries"
print("\nShapes OK")

In [ ]:
# Working data: the full file, or for a dry run a small copy in the same format
if DRY_RUN:
    DATA = REPO / "data" / f"dry_subset_{DRY_TRAIN_ROWS}.h5"
    if not DATA.exists():
        with h5py.File(FULL_DATA, "r") as src, h5py.File(DATA, "w") as dst:
            dst.create_dataset("train", data=src["train"][:DRY_TRAIN_ROWS])
            dst.create_dataset("test", data=src["test"][:])
else:
    DATA = FULL_DATA

with h5py.File(DATA, "r") as f:
    queries = f["test"][:]
    n_train_used = f["train"].shape[0]
    max_dev = 0.0                                   # read train in chunks to spare RAM
    for start in range(0, n_train_used, 50_000):
        norms = np.linalg.norm(f["train"][start:start + 50_000], axis=1)
        max_dev = max(max_dev, float(np.abs(norms - 1).max()))

query_dev = float(np.abs(np.linalg.norm(queries, axis=1) - 1).max())
assert max_dev < 1e-5 and query_dev < 1e-5, "vectors are not unit length"

QUERY_IDS = np.arange(len(queries) if N_QUERIES is None else N_QUERIES)
CONFIG.update({"data_file": DATA.name, "n_train": int(n_train_used), "dim": int(dim),
               "n_queries_run": len(QUERY_IDS)})
save_config()
print(f"Using {DATA.name}: {n_train_used:,} vectors, running {len(QUERY_IDS)} of {len(queries)} queries")
print(f"Largest deviation from length 1: train {max_dev:.1e}, queries {query_dev:.1e}")

In [ ]:
# Build the index (skipped if it already exists; dry runs always rebuild to test the build code)
INDEX = REPO / "save" / (("dry_" if DRY_RUN else "")
                         + f"ccnews_L{LEVELS}_{STRATEGY}_{TARGET_CLUSTER_ITEMS}.zarr")
BUILD_INFO = INDEX.parent / (INDEX.name + ".build.json")    # build stats kept next to the index

if DRY_RUN and INDEX.exists():
    shutil.rmtree(INDEX)

if INDEX.exists():
    print("Index exists, skipping build:", INDEX)
    build_stats = json.loads(BUILD_INFO.read_text()) if BUILD_INFO.exists() else {"note": "built earlier, stats not recorded"}
else:
    INDEX.parent.mkdir(parents=True, exist_ok=True)
    with PeakMemory() as mem:
        t0 = time.perf_counter()
        builder = Builder(index_path=str(INDEX), levels=LEVELS, metric=Metric.L2,
                          is_normalized=True, memory_limit_bytes=BUILD_MEMORY_LIMIT)
        builder.select_representatives(embeddings_file=str(DATA),
                                       target_cluster_items=TARGET_CLUSTER_ITEMS,
                                       strategy=STRATEGY, fallback_batch_rows=10_000,
                                       grp_name="train")
        t_select = time.perf_counter() - t0
        builder.build(embeddings_file=str(DATA), fallback_batch_rows=10_000, grp_name="train")
        t_total = time.perf_counter() - t0
        del builder
    build_stats = {"select_seconds": round(t_select, 1),
                   "build_seconds": round(t_total - t_select, 1),
                   "total_seconds": round(t_total, 1),
                   "ram_peak_mb": mem.peak_mb, "ram_added_mb": mem.added_mb,
                   "built_on": ENV["hostname"], "git_commit": ENV["git_commit"]}
    BUILD_INFO.write_text(json.dumps(build_stats, indent=2))

size_bytes, n_files = folder_size(INDEX)
build_stats.update({"index_size_gb": round(size_bytes / 1e9, 3), "index_files": n_files,
                    "raw_data_gb": round(n_train_used * dim * 4 / 1e9, 3)})
CONFIG.update({"index": INDEX.name, "build": build_stats})
save_config()
build_stats

## Stage 2 · Run queries

 For each `search_exp` (leaves read per query) we open a **new** `Index` (empty cache) and run the queries twice:
- **fresh**: nodes must be read from disk (on Linux we also clear the OS file cache first, so it is a true cold start; on Windows it is only a fresh process);
- **cached**: the same queries again, everything is already in eCP-FS's memory.

eCP-FS is disk-based, so both matter. We only record latency and the returned IDs here; recall is computed in stage 4.

Two things we learned about the library: `close()` saves every open query to disk (slow, so we time it separately and erase them afterwards), and `close()` keeps the node cache in memory until the object is deleted, so we `del` it before the next setting.

In [ ]:
def run_queries(index, search_exp, k=K, max_increments=-1):
    # Runs every query in QUERY_IDS once. One row per query: latency + returned ids.
    # max_increments=-1 lets the search widen until it has k items, so every query returns k.
    rows = []
    for qi in QUERY_IDS:
        t0 = time.perf_counter()
        results, _ = index.new_search(query=queries[qi], k=k, search_exp=search_exp,
                                      max_increments=max_increments, exclude_vec=[])
        ms = (time.perf_counter() - t0) * 1000
        rows.append({"query": int(qi), "latency_ms": ms, "ids": [int(i) for _, i in results]})
    assert len(rows) == len(QUERY_IDS), "wrong number of queries ran"
    return rows

def clear_saved_queries():
    # Erase the query states close() wrote, so runs don't pile up in the index
    with Index(str(INDEX)) as idx:
        return idx.cleanup_persisted_queries_older_than(int(time.time()) + 1)

In [ ]:
# Stage 2a: the search_exp sweep
free_gb = psutil.virtual_memory().available / 1e9
print(f"Free RAM before the sweep: {free_gb:.1f} GB")
assert DRY_RUN or free_gb > 2, "Less than 2 GB free: close other programs and kernels first"
raw, runs = [], []
t_sweep = time.perf_counter()

for rep in range(REPEATS):
    for se in SWEEP:
        evicted = EVICT_OS_CACHE and evict_os_cache(INDEX)
        with PeakMemory() as mem:
            index = Index(str(INDEX), memory_limit_bytes=SEARCH_MEMORY_LIMIT)  # new Index = empty cache
            passes = {"fresh": run_queries(index, se), "cached": run_queries(index, se)}
            t0 = time.perf_counter(); index.close(); close_s = time.perf_counter() - t0
            del index; gc.collect()                    # close() keeps the cache; deleting frees it
        t0 = time.perf_counter(); erased = clear_saved_queries(); cleanup_s = time.perf_counter() - t0

        for pass_name, rows in passes.items():
            raw += [{"rep": rep, "search_exp": se, "pass": pass_name, **r} for r in rows]
        runs.append({"rep": rep, "search_exp": se, "os_cache_evicted": evicted,
                     "ram_start_mb": mem.start_mb, "ram_peak_mb": mem.peak_mb,
                     "ram_added_mb": mem.added_mb, "close_s": round(close_s, 1),
                     "cleanup_s": round(cleanup_s, 1), "queries_erased": erased,
                     "free_ram_gb": round(psutil.virtual_memory().available / 1e9, 2)})
        fresh_med = np.median([r["latency_ms"] for r in passes["fresh"]])
        print(f"rep {rep}  search_exp={se:4d}  fresh median {fresh_med:8.1f} ms  "
              f"close {close_s:6.1f} s  cleanup {cleanup_s:6.1f} s  RAM +{mem.added_mb} MB")

# Raw results: one row per (repeat, search_exp, pass, query). ids are stored as "id;id;..."
raw_df = pd.DataFrame(raw)
raw_df.assign(ids=raw_df["ids"].map(lambda ids: ";".join(map(str, ids)))).to_csv(
    OUT / "raw_results.csv", index=False)
runs_df = pd.DataFrame(runs)
runs_df.to_csv(OUT / "runs.csv", index=False)
CONFIG["sweep_minutes"] = round((time.perf_counter() - t_sweep) / 60, 1)
save_config()
runs_df

In [ ]:
# Stage 2b (optional): is cached latency really flat? One cache-capped index, warmed up with the
# largest search_exp. Best-first search is deterministic, so a smaller search_exp reads a prefix
# of the same leaves: every measured pass below is fully cached.
if RUN_CACHE_TEST:
    cache_rows = []
    index = Index(str(INDEX), memory_limit_bytes=SEARCH_MEMORY_LIMIT)
    run_queries(index, max(SWEEP))                       # warm-up, not measured
    for se in SWEEP:
        for rep in range(3):
            pf0 = page_faults()
            lat = np.array([r["latency_ms"] for r in run_queries(index, se)])
            cache_rows.append({"search_exp": se, "rep": rep, "mean_ms": lat.mean(),
                               "median_ms": np.median(lat), "p99_ms": np.percentile(lat, 99),
                               "max_ms": lat.max(), "page_faults": page_faults() - pf0,
                               "ram_mb": ram_mb(),
                               "free_ram_gb": psutil.virtual_memory().available / 1e9})
    index.close(); del index; gc.collect()
    clear_saved_queries()
    cache_test = pd.DataFrame(cache_rows).round(2)
    cache_test.to_csv(OUT / "cache_test.csv", index=False)
    display(cache_test.groupby("search_exp")[["mean_ms", "median_ms", "p99_ms",
                                              "page_faults", "free_ram_gb"]].median())
else:
    print("Skipped (RUN_CACHE_TEST = False)")

## Stage 3 · Ground truth

 The ground truth is the *answer key*: each query's true nearest neighbours, found by comparing it with **every** vector (brute force). For unit vectors, ‖q − x‖² = 2 − 2·q·x, so ranking by dot product (highest first) is the same as ranking by L2 (lowest first), and the dot product is the cheapest to compute: one matrix multiplication per chunk. We verify our result against the answer key that ships with the dataset; it is computed once and then loaded from disk.

In [ ]:
K_GT = 100
GT_DIR = REPO / "results" / (f"ground_truth_dry_{DRY_TRAIN_ROWS}" if DRY_RUN else "ground_truth")

def brute_force_top_k(data_file, queries, k=K_GT, chunk=50_000):
    # Exact top-k by dot product, reading train in chunks; returns (ids, cosine distances)
    best_sim = np.full((len(queries), k), -np.inf, dtype=np.float32)
    best_ids = np.full((len(queries), k), -1, dtype=np.int64)
    with h5py.File(data_file, "r") as f:
        n = f["train"].shape[0]
        for start in range(0, n, chunk):
            sims = queries @ f["train"][start:start + chunk].T          # (queries, chunk)
            top = np.argpartition(-sims, k, axis=1)[:, :k]               # best k in this chunk
            cand_sim = np.concatenate([best_sim, np.take_along_axis(sims, top, axis=1)], axis=1)
            cand_ids = np.concatenate([best_ids, top + start], axis=1)   # local -> global id
            keep = np.argpartition(-cand_sim, k, axis=1)[:, :k]          # merge with best so far
            best_sim = np.take_along_axis(cand_sim, keep, axis=1)
            best_ids = np.take_along_axis(cand_ids, keep, axis=1)
    order = np.argsort(-best_sim, axis=1)
    return np.take_along_axis(best_ids, order, axis=1), 1 - np.take_along_axis(best_sim, order, axis=1)

if (GT_DIR / "gt_ids.npy").exists():
    gt_ids = np.load(GT_DIR / "gt_ids.npy")
    gt_meta = json.loads((GT_DIR / "meta.json").read_text()) if (GT_DIR / "meta.json").exists() else {}
    print("Loaded saved ground truth from", GT_DIR)
else:
    t0 = time.perf_counter()
    gt_ids, gt_dist = brute_force_top_k(DATA, queries)
    seconds = time.perf_counter() - t0
    gt_meta = {"brute_force_seconds": round(seconds, 1),
               "brute_force_ms_per_query": round(seconds / len(queries) * 1000, 2),
               "computed_on": ENV["hostname"]}
    if not DRY_RUN:                       # the file's answer key only matches the full data
        with h5py.File(FULL_DATA, "r") as f:
            file_ids, file_dist = f["neighbors"][:], f["distances"][:]
        gt_meta["top100_agreement_with_file"] = float(np.mean(
            [len(set(a) & set(b)) / K_GT for a, b in zip(gt_ids, file_ids)]))
        gt_meta["max_distance_difference"] = float(np.abs(gt_dist - file_dist).max())
        assert gt_meta["top100_agreement_with_file"] > 0.999, "our ground truth disagrees with the file"
    GT_DIR.mkdir(parents=True, exist_ok=True)
    np.save(GT_DIR / "gt_ids.npy", gt_ids)
    np.save(GT_DIR / "gt_dist.npy", gt_dist)
    (GT_DIR / "meta.json").write_text(json.dumps(gt_meta, indent=2))
    print(f"Brute force took {seconds:.1f} s; saved to {GT_DIR}")

CONFIG["ground_truth"] = gt_meta
save_config()
gt_meta

## Stage 4 · Evaluate

 **Recall@10** = how many of the 10 true neighbours eCP-FS returned, divided by 10. Speed is reported as latency per query; the **median** and **p99** are more honest than the mean, which a few slow outliers (e.g. page faults) can pull up. **QPS** = 1000 / mean latency (single-threaded). Quality and speed are always reported together, because each can be bought with the other: the recall-vs-QPS curve is what  VIBE compares.

To re-evaluate an older run, point `EVAL_DIR` at its folder.

In [ ]:
EVAL_DIR = OUT

res = pd.read_csv(EVAL_DIR / "raw_results.csv", dtype={"ids": str})
res["ids"] = res["ids"].fillna("").map(lambda s: [int(x) for x in s.split(";") if x])

def recall_at_k(ids, true_ids, k=K):
    return len(set(ids[:k]) & set(true_ids[:k].tolist())) / k

res["recall"] = [recall_at_k(ids, gt_ids[q]) for ids, q in zip(res["ids"], res["query"])]
res["returned"] = res["ids"].map(len)

summary = (res.groupby(["search_exp", "pass"])
              .agg(recall=("recall", "mean"),
                   mean_ms=("latency_ms", "mean"),
                   median_ms=("latency_ms", "median"),
                   p99_ms=("latency_ms", lambda x: np.percentile(x, 99)),
                   min_returned=("returned", "min"),
                   n=("query", "count"))
              .reset_index())
summary["qps"] = 1000 / summary["mean_ms"]
summary = summary.round({"recall": 4, "mean_ms": 2, "median_ms": 2, "p99_ms": 2, "qps": 1})
summary.to_csv(EVAL_DIR / "summary.csv", index=False)

assert (summary["min_returned"] == K).all(), "some queries returned fewer than k results"
summary.pivot(index="search_exp", columns="pass",
              values=["recall", "median_ms", "p99_ms", "qps"])

In [ ]:
fresh = summary[summary["pass"] == "fresh"]
cached = summary[summary["pass"] == "cached"]
bf_ms = CONFIG.get("ground_truth", {}).get("brute_force_ms_per_query")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

ax[0].plot(fresh["search_exp"], fresh["recall"], "o-")
ax[0].set(xscale="log", xlabel="search_exp (leaves read)", ylabel="Recall@10", ylim=(0, 1),
          title="Quality")
ax[0].set_xticks(SWEEP, labels=[str(s) for s in SWEEP])

for df_, label in [(fresh, "fresh"), (cached, "cached")]:
    ax[1].plot(df_["search_exp"], df_["median_ms"], "o-", label=f"{label} (median)")
if bf_ms:
    ax[1].axhline(bf_ms, ls="--", c="gray", label="brute force (exact)")
ax[1].set(xscale="log", yscale="log", xlabel="search_exp", ylabel="ms per query", title="Latency")
ax[1].set_xticks(SWEEP, labels=[str(s) for s in SWEEP])
ax[1].legend()

for df_, label in [(fresh, "fresh"), (cached, "cached")]:
    ax[2].plot(df_["recall"], df_["qps"], "o-", label=label)
ax[2].set(yscale="log", xlabel="Recall@10", ylabel="queries per second", xlim=(0, 1),
          title="Trade-off (up and right is better)")
ax[2].legend()

fig.suptitle(f"eCP-FS on {DATASET}" + ("  [DRY RUN]" if DRY_RUN else ""))
fig.tight_layout()
fig.savefig(EVAL_DIR / "plots.png", dpi=150)
plt.show()

In [ ]:
save_config()
print("Run folder:", OUT)
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:22s} {p.stat().st_size / 1e3:8.1f} KB")

## Stage 5 · Deeper analysis



### 5A · Where do the true neighbours live?

 eCP-FS can only find a true neighbour if it opens the leaf that neighbour lives in. So for each query we look up the leaf of each of its 10 true neighbours, and ask: if we ranked all 11,520 leaves by how similar their leader is to the query, how far down that list is the neighbour's leaf?

That gives a recall curve for a search that compares the query with **all** leaders instead of walking down the tree. Comparing it with eCP-FS's measured curve separates two possible causes of low recall:
- **Big gap between the curves:** the tree's routing loses neighbours. Fewer levels should help.
- **Small gap, both low:** the clusters themselves scatter neighbours over many leaves. Leaf size or how leaders are chosen matters more.

It's not a strict upper limit, because the build also assigned each article to its leaf through the tree, so eCP-FS can occasionally beat it slightly. It also estimates how many leaves 90% recall would need, without running those big searches.

In [ ]:
import zarr

root = zarr.open_group(str(INDEX), mode="r")
INDEX_LEVELS = int(root["info/levels"][()])
leaf_group = root[f"lvl_{INDEX_LEVELS}"]

t0 = time.perf_counter()
# 1) Which leaf holds each article, and how many articles each leaf holds
item_leaf = np.full(int(root["info/next_item_id"][()]), -1, dtype=np.int64)
leaf_size = {}
for name in leaf_group.group_keys():
    leaf = int(name.split("_")[1])
    ids = leaf_group[name]["item_ids"][:]
    item_leaf[ids] = leaf
    leaf_size[leaf] = len(ids)

# 2) The leader of each leaf. A leaf's leader is stored in its parent node one level up:
#    'embeddings' row i is the leader of the leaf named in 'node_ids' row i
if INDEX_LEVELS == 1:
    leader_emb = root["index_root/embeddings"][:].astype(np.float32)
    leader_leaf = np.arange(len(leader_emb))
else:
    parent_group = root[f"lvl_{INDEX_LEVELS - 1}"]
    embs, leaves = [], []
    for name in parent_group.group_keys():
        node = parent_group[name]
        embs.append(node["embeddings"][:])
        leaves.append(node["node_ids"][:])
    leader_emb = np.concatenate(embs).astype(np.float32)
    leader_leaf = np.concatenate(leaves).astype(np.int64)

assert (item_leaf >= 0).all(), "some articles are in no leaf"
assert sum(leaf_size.values()) == len(item_leaf), "some articles are in more than one leaf"
print(f"Read {len(leaf_size):,} leaves and {len(leader_leaf):,} leaders "
      f"in {time.perf_counter() - t0:.0f} s; every article is in exactly one leaf")

In [ ]:
B_GRID = [2**i for i in range(13)]                      # 1, 2, 4 … 4,096 leaves
qs = queries[QUERY_IDS]

# Rank every leaf for every query by how similar its leader is (rank 0 = most similar leader)
order = np.argsort(-(qs @ leader_emb.T), axis=1)        # leader positions, best first
leaf_rank = np.full((len(qs), int(leader_leaf.max()) + 1), np.iinfo(np.int32).max, dtype=np.int32)
leaf_rank[np.arange(len(qs))[:, None], leader_leaf[order]] = np.arange(order.shape[1])

nb_leaf = item_leaf[gt_ids[QUERY_IDS, :K]]                      # leaf of each true neighbour
nb_rank = np.take_along_axis(leaf_rank, nb_leaf, axis=1)        # its leaf's place in that ranking

# Recall if a search opened the b leaves with the most similar leaders (no tree in the way)
flat = pd.DataFrame({"search_exp": B_GRID, "recall_flat": [(nb_rank < b).mean() for b in B_GRID]})
measured = (summary[summary["pass"] == "fresh"][["search_exp", "recall"]]
            .rename(columns={"recall": "recall_ecpfs"}))
neighbour_map = flat.merge(measured, on="search_exp", how="left")
neighbour_map["routing_loss"] = neighbour_map["recall_flat"] - neighbour_map["recall_ecpfs"]

ranks = np.sort(nb_rank.ravel())
b90 = int(ranks[int(np.ceil(0.9 * len(ranks))) - 1]) + 1       # smallest b that reaches 90%
CONFIG.update({"b_for_90pct_recall_flat": b90, "n_leaves": len(leaf_size)})
save_config()
neighbour_map.to_csv(OUT / "neighbour_map.csv", index=False)

print(f"Same leaf as the query's most similar leader: {(nb_rank == 0).mean():.1%} of true neighbours")
print(f"Leaves needed for 90% recall without the tree: b = {b90:,} "
      f"({b90 / len(leaf_size):.1%} of all {len(leaf_size):,} leaves)")
neighbour_map.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(neighbour_map["search_exp"], neighbour_map["recall_flat"], "o-",
        label="no tree: compare the query with all leaders")
m = neighbour_map.dropna(subset=["recall_ecpfs"])
ax.plot(m["search_exp"], m["recall_ecpfs"], "s-", label="eCP-FS, measured (tree)")
ax.axhline(0.9, ls="--", c="gray", lw=1)
ax.axvline(b90, ls=":", c="gray", lw=1)
ax.text(b90, 0.03, f" b = {b90:,}", color="gray")
ax.set(xscale="log", xlabel="search_exp b (leaves opened)", ylabel="Recall@10", ylim=(0, 1),
       title="Where do the true neighbours live?")
ax.set_xticks(B_GRID, labels=[str(b) for b in B_GRID])
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "neighbour_map.png", dpi=150)
plt.show()

### 5D · Leaf sizes

 The build aims for 43 articles per leaf, but eCP doesn't enforce it: every article simply joins its leader's leaf. Uneven leaves matter for speed: a query that opens a leaf of 300 articles does 7× the work of one that opens a leaf of 43. That's one reason some queries are much slower than others.

In [ ]:
sizes = np.array(list(leaf_size.values()))
target = CONFIG["target_cluster_items"]
big = sizes > 2 * target
print(f"{len(sizes):,} leaves · target {target} · mean {sizes.mean():.1f} · median {np.median(sizes):.0f} "
      f"· smallest {sizes.min()} · largest {sizes.max()}")
print(f"{big.mean():.1%} of leaves hold more than twice the target, "
      f"and they contain {sizes[big].sum() / sizes.sum():.1%} of all articles")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(sizes, bins=60)
ax.axvline(target, ls="--", c="gray", label=f"target ({target})")
ax.set(xlabel="articles in the leaf", ylabel="number of leaves", title="Leaf sizes")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "leaf_sizes.png", dpi=150)
plt.show()
CONFIG["leaf_sizes"] = {"mean": float(sizes.mean()), "median": float(np.median(sizes)),
                        "max": int(sizes.max()), "share_over_2x_target": float(big.mean())}
save_config()

### 5B · Recall the way  VIBE counts it

 So far, recall counted exact ID matches. VIBE counts a result as correct if its **distance** is no larger than the 10th true neighbour's distance plus a small tolerance (ε = 0.001). That way an article that is practically tied with a true neighbour isn't counted as a miss. We compute both, at k = 1 and k = 10, so our numbers can be compared directly with theirs. The distances are recomputed from the returned IDs, so no new search is needed.

In [ ]:
EPS = 1e-3
gt_dist = np.load(GT_DIR / "gt_dist.npy")                # true cosine distances, best first

# Distances of the returned articles: read train once, in chunks, keeping only the rows we need
needed = np.unique(np.concatenate(res["ids"].values))
rows = np.empty((len(needed), queries.shape[1]), dtype=np.float32)
with h5py.File(DATA, "r") as f:
    for start in range(0, f["train"].shape[0], 50_000):
        sel = np.nonzero((needed >= start) & (needed < start + 50_000))[0]
        if len(sel):
            rows[sel] = f["train"][start:start + 50_000][needed[sel] - start]

def found_dist(ids, q):
    # cosine distance from query q to each returned article, in the order eCP-FS returned them
    return 1 - rows[np.searchsorted(needed, ids)] @ queries[q]

def recalls(ids, q):
    d = found_dist(ids, q)
    return {"recall@1_id": float(ids[0] == gt_ids[q, 0]),
            "recall@1_dist": float(d[0] <= gt_dist[q, 0] + EPS),
            "recall@10_id": len(set(ids[:10]) & set(gt_ids[q, :10].tolist())) / 10,
            "recall@10_dist": min(1.0, float(np.sum(d[:10] <= gt_dist[q, 9] + EPS)) / 10)}

fresh_res = res[res["pass"] == "fresh"]
variants = pd.DataFrame([recalls(ids, q) for ids, q in zip(fresh_res["ids"], fresh_res["query"])],
                        index=fresh_res.index)
variants["search_exp"] = fresh_res["search_exp"]
recall_variants = variants.groupby("search_exp").mean().round(4)
recall_variants.to_csv(OUT / "recall_variants.csv")
recall_variants

### 5C · Which queries are hard?

 The average recall hides a lot: some queries get 10/10, others 0/10. A common measure of how hard a query is, is its **relative contrast**: the average distance to all articles divided by the distance to its 10th true neighbour. High contrast means the neighbours stand out clearly (easy). Contrast near 1 means everything is about equally far away (hard for any index). The dataset stores the average distance per query as `avg_distances`. If eCP-FS mostly fails on low-contrast queries, part of the low recall is a property of the data, not of eCP-FS.

In [ ]:
with h5py.File(FULL_DATA, "r") as f:
    avg_dist = f["avg_distances"][:]
# Relative contrast: average distance to everything / distance to the 10th true neighbour.
# High = the neighbours stand out clearly (easy); close to 1 = everything is about as far (hard).
rc = avg_dist / gt_dist[:, K - 1]

SE_C = max(SWEEP)
pq = (res[(res["pass"] == "fresh") & (res["search_exp"] == SE_C) & (res["rep"] == 0)]
      .set_index("query")["recall"])
hard = pd.DataFrame({"recall": pq, "relative_contrast": rc[pq.index]})
hard["difficulty"] = pd.qcut(hard["relative_contrast"], 4,
                             labels=["hardest 25%", "harder", "easier", "easiest 25%"])
hard.to_csv(OUT / "query_difficulty.csv")

zero_share = (res[(res["pass"] == "fresh") & (res["rep"] == 0)]
              .groupby("search_exp")["recall"].apply(lambda r: (r == 0).mean()))
print("Share of queries with recall 0, per search_exp:")
print(zero_share.map("{:.1%}".format).to_string())
print(f"\nRank correlation between relative contrast and recall at search_exp={SE_C}: "
      f"{hard['relative_contrast'].rank().corr(hard['recall'].rank()):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(hard["recall"], bins=np.arange(0, 1.11, 0.1) - 0.05, rwidth=0.85)
ax[0].set(xlabel="Recall@10 of one query", ylabel="number of queries",
          title=f"Per-query recall at search_exp={SE_C}")
by_q = hard.groupby("difficulty", observed=True)["recall"].mean()
ax[1].bar(by_q.index.astype(str), by_q.values)
ax[1].set(ylabel="mean Recall@10", ylim=(0, 1), title="Recall by query difficulty (relative contrast)")
fig.tight_layout()
fig.savefig(OUT / "query_difficulty.png", dpi=150)
plt.show()

### 5F · Error bars

 With 200 queries, every recall number is an estimate. The **bootstrap** checks how much it could move: draw 200 queries at random *with replacement* from the ones we have, compute recall, repeat 2,000 times. The middle 95% of those results is the 95% confidence interval. If two settings' intervals overlap a lot, the difference between them may not be real. This matters as soon as we compare build settings.

In [ ]:
rng = np.random.default_rng(0)

def bootstrap_ci(values, n_boot=2000):
    values = np.asarray(values)
    means = rng.choice(values, size=(n_boot, len(values))).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

ci_rows = []
for se, r in res[(res["pass"] == "fresh") & (res["rep"] == 0)].groupby("search_exp")["recall"]:
    low, high = bootstrap_ci(r)
    ci_rows.append({"search_exp": se, "recall": r.mean(), "low_95": low, "high_95": high,
                    "plus_minus": (high - low) / 2})
recall_ci = pd.DataFrame(ci_rows).round(4)
recall_ci.to_csv(OUT / "recall_ci.csv", index=False)
recall_ci